# New Section

In [34]:
%pip install -U langchain langchain-community langchain-huggingface chromadb pymupdf sentence-transformers

In [28]:
%pip install -U "attrs>=23.2.0" "jsonschema>=4.23.0" chromadb

In [29]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pymupdf
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

In [30]:
pdf=pymupdf.open("/content/The+48+Laws+Of+Power.pdf")
print("total number of pages in the pdf are :",len(pdf))
page=pdf[0]
print(page)

total number of pages in the pdf are : 476
page 0 of </content/The+48+Laws+Of+Power.pdf, doc# 4>


In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
spiltter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)

from langchain_core.documents import Document
documents=[]
for page_number,page in enumerate(pdf):
  text=page.get_text()
  if text.strip():
    documents.append(
        Document(page_content=text,
                 metadata={"page":page_number+1}
        )
    )
chunks=spiltter.split_documents(documents)
print("number of chunks:",len(chunks))



number of chunks: 3203


In [32]:
!pip install -U langchain-chroma

In [25]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="rag_collection",
    embedding_function=embeddings
)

vector_store.add_documents(chunks)

print("Vector store created successfully!")

Vector store created successfully!


In [26]:
#lext creat embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector = embeddings.embed_query(chunks[0].page_content)

print("Vector dimensions:", len(vector))
print("First 10 values:", vector[:10])
print("Documents added to ChromaDB!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector dimensions: 384
First 10 values: [-0.028905833140015602, 0.06803185492753983, 0.00815645419061184, 0.06961315870285034, -0.023174677044153214, 0.01907346583902836, -0.02742033265531063, -0.07621227204799652, -0.011454537510871887, 0.08192763477563858]
Documents added to ChromaDB!


In [36]:
#retrival
#now basicall what we did was we imported the pdf kept in the documents ,split them into chunks,emmbed them into vecotrs stored in chroma database.
#now this part we are going to retrive it
queary="what was this document about"
results=vector_store.similarity_search(queary,k=3)#for retrival we are not using any llm insted we use huggingface retrival that retrives the similar chunks related yo the question

for i, result in enumerate(results):

    print(f"\n--- Result {i + 1} ---")
    print(result.page_content)
    print("Page:", result.metadata["page"])


--- Result 1 ---
New York: Penguin Books, 1976.
Clausewitz, Carl von. On War. Edited and translated by Michael Howard
and Peter Paret. Princeton: Princeton University Press, 1976.
Elias, Norbert. 7713 Court Society. Translated by Edmund jephcott. Oxford:
Basil Blackwell Publishers, 1983.
de Francesco, Grete.
The Power of the Charlatan. Translated by Miriam
Beard. New Haven: Yale University Press, 1939.
Haley, Jay.
The Power Ihttics ofJesus Christ and Other Essays. New York:
W. W. Norton. 1989.
Page: 454

--- Result 2 ---
loosely tied papers and books-“Yes,” he replied, “if you do not walk too
fast.”
As they started out, however, the sun was setting. Afraid of being
locked out of the town at night, prey to local bandits, Chou and the boy
walked faster and faster, finally breaking into a run. Suddenly the string
around the papers broke and the documents scattered on the ground. It
took them many minutes to put the packet together again, and by the time
they had reached the city gates, i